**Delivery Optimisation Linear Programming**


In [20]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

# --- 1. PARAMETERS ---
P = 5.00
depot_labels = ["D1", "D2", "D3"]
store_labels = ["Store 1", "Store 2", "Store 3"]

distances = np.array([
    [22, 33, 40],
    [27, 30, 22],
    [36, 20, 25],
])

# Inventory and Capacity
supply_fixed = [2500, 3100, 1250]  # Total 6850 (Must be 100% delivered)
store_caps = [2000, 3000, 2000]    # Total 7000 (The "Limit")

# --- 2. OBJECTIVE FUNCTION ---
# Z = sum(x_ij * D_ij * P)
c = (distances * P).flatten()

# --- 3. CONSTRAINTS ---
# Depot Supply: Sum of units FROM each depot MUST EQUAL fixed supply
A_eq = np.zeros((3, 9))
for i in range(3):
    A_eq[i, 3*i : 3*i+3] = 1
b_eq = supply_fixed

# Store Capacity: Sum of units TO each store MUST BE <= capacity
A_ub = np.zeros((3, 9))
for j in range(3):
    for i in range(3):
        A_ub[j, 3*i + j] = 1
b_ub = store_caps

# --- 4. SOLVER ---
res = linprog(
    c=c,
    A_ub=A_ub, b_ub=b_ub,
    A_eq=A_eq, b_eq=b_eq,
    method="highs"
)

# --- 5. RESULTS ---
if res.success:
    optimal_x = res.x.reshape(3, 3).round(0)
    df = pd.DataFrame(optimal_x, index=depot_labels, columns=store_labels)

    print(f"Total Minimum Cost (Z): £{res.fun:,.2f}")
    print("\nOptimal Shipping Matrix (6,850 units delivered):")
    print(df)

    # Validation of your Slack logic
    print("\n" + "="*30)
    print("CAPACITY VS ACTUAL DELIVERY")
    summary = pd.DataFrame({
        "Capacity": store_caps,
        "Delivered": optimal_x.sum(axis=0),
        "Slack": np.array(store_caps) - optimal_x.sum(axis=0)
    }, index=store_labels)
    print(summary)

Total Minimum Cost (Z): £812,500.00

Optimal Shipping Matrix (6,850 units delivered):
    Store 1  Store 2  Store 3
D1   2000.0    500.0      0.0
D2      0.0   1100.0   2000.0
D3      0.0   1250.0      0.0

CAPACITY VS ACTUAL DELIVERY
         Capacity  Delivered  Slack
Store 1      2000     2000.0    0.0
Store 2      3000     2850.0  150.0
Store 3      2000     2000.0    0.0
